# task_22 Solution

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_score, recall_score

base = Path("../../tasks/task_22/data/")

In [ ]:
customers = pd.read_csv(base / "customers.csv")
split = pd.read_csv(base / "split.csv")

df = customers.merge(split, on="customer_id")
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

numeric_features = ["tenure_months", "monthly_charges", "total_charges",
                    "num_support_tickets", "has_online_backup", "has_streaming",
                    "senior_citizen", "partner"]
categorical_features = ["contract_type", "internet_service"]
target = "churned"

# Encode categoricals with drop_first=True
train_enc = pd.get_dummies(train, columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test, columns=categorical_features, drop_first=True)

# Align columns (in case test is missing a dummy level)
test_enc = test_enc.reindex(columns=train_enc.columns, fill_value=0)

# All feature columns (numeric + encoded dummies)
feature_cols = [c for c in train_enc.columns if c not in ["customer_id", "split", target]]
print("Features:", feature_cols)

Q1: Among the original numeric features only, which has the highest odds ratio in the unscaled logistic regression model?

In [ ]:
# Fit unscaled logistic regression on all features
lr_unscaled = LogisticRegression(max_iter=1000, random_state=42)
lr_unscaled.fit(train_enc[feature_cols], train_enc[target])

# Compute odds ratios for numeric features only
coef_map = dict(zip(feature_cols, lr_unscaled.coef_[0]))
odds_ratios = {feat: np.exp(coef_map[feat]) for feat in numeric_features}

q1 = max(odds_ratios, key=odds_ratios.get)
print("Odds ratios (numeric features):")
for feat, or_val in sorted(odds_ratios.items(), key=lambda x: -x[1]):
    print(f"  {feat}: {or_val:.4f}")
print(f"\nQ1 answer: {q1}")

Q2: Fit standardized logistic regression on all features. Report AUC-ROC on test, rounded to 2 decimal places.

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_enc[feature_cols])
X_test_scaled = scaler.transform(test_enc[feature_cols])

y_train = train_enc[target].values
y_test = test_enc[target].values

# Fit standardized logistic regression
lr_scaled = LogisticRegression(max_iter=1000, random_state=42)
lr_scaled.fit(X_train_scaled, y_train)

# Predict probabilities on test
y_prob = lr_scaled.predict_proba(X_test_scaled)[:, 1]

auc = roc_auc_score(y_test, y_prob)
q2 = round(auc, 2)
print(f"Q2 answer: {q2}")

Q3: Predict on test at 0.5 threshold. How many false negatives?

In [ ]:
# Predict at default 0.5 threshold
y_pred = (y_prob >= 0.5).astype(int)

# Confusion matrix: [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_test, y_pred)
fn = cm[1, 0]  # actual=1, predicted=0

q3 = int(fn)
print(f"Confusion matrix:\n{cm}")
print(f"\nQ3 answer: {q3}")

Q4: Highest threshold (nearest 0.05) achieving >= 80% recall on test.

In [ ]:
# Sweep thresholds from 0.05 to 0.95 in steps of 0.05
thresholds = np.arange(0.05, 1.0, 0.05)
best_threshold = None

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    rec = recall_score(y_test, y_pred_t)
    if rec >= 0.80:
        best_threshold = round(t, 2)

q4 = best_threshold
print(f"Q4 answer: {q4}")

Q5: Precision at 0.5 threshold on test, rounded to 2 decimal places.

In [ ]:
# Precision at 0.5 threshold (already computed y_pred above)
prec = precision_score(y_test, y_pred)
q5 = round(prec, 2)
print(f"Q5 answer: {q5}")